# OSU DEMO

This file expects to be in chipwhisperer/jupyter/courses/fault101. If you opened this in vscode, your local directory might be different. you can either cd' inside this notebook (not in the terminal) to where it expects to be, or adjust the other commands. Specifically, the compiling and programming steps below

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_NEORV32'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "./jupyter/Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

In [3]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ./firmware/mcu/simpleserial-glitch-tiny-regex
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j OPT=0

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
riscv64-unknown-elf-gcc (g04696df09) 14.2.0
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CW308_NEORV32 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
-en     test1.c ...
-en     .././simpleserial/simpleserial.c ...
Compiling:
-en     .././hal/hal.c ...
-en     .././hal//neorv32/neorv32_cfs.c ...
-en     test2.c ...
Compiling:
Compiling:
-en     simpleserial-glitch-tiny-regex.c ...
-en     re.c ...
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
-en     .././hal//neorv32/neorv32_gpio.c ...
-en     .././hal//neorv32/neorv32_mtime.c ...
.
-en     .././hal//neorv32/neorv32_gptmr.c ...
Compiling:
-en     .././hal//neorv32/neorv32_cpu.c ...
.
.
.
-en     .././hal//neorv32/neorv32_neoled.c ...
Compil

.././hal//neorv32/syscalls.c:104:6: warning: "/*" within comment [-Wcomment]
  104 |     //*(volatile int *)EXIT_REG = exit_status;


-e Done!


.././hal//neorv32/syscalls.c:135:19: warning: 'struct timeb' declared inside parameter list will not be visible outside of this definition or declaration
  135 | int _ftime(struct timeb *tp)
      |                   ^~~~~
.././hal//neorv32/syscalls.c: In function '_sbrk':
.././hal//neorv32/syscalls.c:267:22: warning: comparison between two arrays [-Warray-compare]
  267 |     if (__heap_start == __heap_end) {
      |                      ^~
.././hal//neorv32/syscalls.c:267:22: note: use '&__heap_start[0] == &__heap_end[0]' to compare the addresses


-e Done!
-e Done!
-e Done!
.
Assembling: .././hal//neorv32/crt0.S
riscv64-unknown-elf-gcc -c -march=rv32i_zicsr -mabi=ilp32  -I. -x assembler-with-cpp -Wall -ffunction-sections -fdata-sections -nostartfiles -mno-fdiv -Wl,--gc-sections -lm -lc -lgcc -lc -falign-functions=4 -falign-labels=4 -falign-loops=4 -falign-jumps=4 -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CW308_NEORV32/crt0.lst -I.././simpleserial/ -I.././hal/ -I.././hal/ -I.././hal//neorv32 -I.././simpleserial/ -I.././crypto/ .././hal//neorv32/crt0.S -o objdir-CW308_NEORV32/crt0.o
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     simpleserial-glitch-tiny-regex-CW308_NEORV32.elf ...
Memory region         Used Size  Region Size  %age Used
             ram:        7136 B        64 KB     10.89%
             rom:       18200 B        64 KB     27.77%
           iodev:           0 B        512 B      0.00%
-e Done!
.
.
.
.
.
Creating load file for Flash: simpleserial-glitch-tiny-regex-CW308_NEORV32.hex
riscv64-unknown-elf-objcopy -O ihe

In [4]:
# Flash program, init comms.
fw_path = "./firmware/mcu/simpleserial-glitch-tiny-regex/simpleserial-glitch-tiny-regex-{}.bin".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

In [5]:
# Define reboot function. This is different for the ICE40.
def reboot_flush():
    #reset_target(scope) # <-- use this for other boards!
    # no reboot on the ICE40 since it doesn't have a way to reset it externally. We just need to reprogram it.
    cw.program_target(scope, prog, fw_path) 
    #Flush garbage too
    target.flush()

In [6]:
# Set default glitch parameters.
scope.cglitch_setup()

In [7]:
# Define define graph parameters
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

In [8]:
# Define graph.
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="ext_offset", y_index="offset")

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [ext_offset,offset]
      .Points.II :Points   [ext_offset,offset]

In [ ]:
from tqdm.notebook import tqdm
import re
import struct
import time

# Number of runs test we test at each glitch configuration.
sample_size = 1

# Range of cycles we will search around target cycle.
ERROR_TOLERANCE = 5#20 

# Target cycle calculated from RTL simulation.
IF_CONDITION = 565637

# Cycles spent bootstrapping and initializing I/O. Chipwhisperer does not count that time. So, subtract that time
MAIN_START = 14339

# Chipwhisperer adds two instructions before the funciton call we are simulating. So, Add that time.
# Also, add time to account for the trigger calls. Exact time is unknown for lowering, assume similar to raising = 45.
CYCLES_FOR_STORE = 6  # Cycles for a storing a word, i.e., 'sw'
CYCLES_FOR_ADD = 2    # Cycles to add an immediate, i.e., 'addi'

# Accruracy for estimations seems to aim a little high for samples on CW, so, adjust it down.
AVERAGE_ERROR = -50.167
OFFSET = int(CYCLES_FOR_STORE + CYCLES_FOR_ADD + AVERAGE_ERROR)


# Note 1: Cycle count found running program and dividing the scope.adc.trig_count, after trigger has been lowered, by 4.
# Note 2: The ADC increments four times each cycle by default. If your numbers are very off, check!
PROGRAM_AVERAGE_CYCLE_COUNT = 463309             # Number of cycles a normal execution takes.
SIMULATION_MAX_CYCLE_COUNT = 565795 - MAIN_START # Number of cycles a simulated execution takes.
SIMULATION_IF_CYCLE_COUNT = IF_CONDITION - MAIN_START + OFFSET
TIME_SCALAR = PROGRAM_AVERAGE_CYCLE_COUNT/SIMULATION_MAX_CYCLE_COUNT # Quotient of the two program runs.
INVERSE_SCALE = SIMULATION_MAX_CYCLE_COUNT/PROGRAM_AVERAGE_CYCLE_COUNT
TARGET_CYCLE = 463160#int((SIMULATION_IF_CYCLE_COUNT) * TIME_SCALAR)
    
# Set up glitch parameters ranges and step.
gc.set_range("width", 3500, 4500)        # Default values from the solution script
#gc.set_range("offset", 2000, 3200)       # Default values from the solution script
gc.set_range("offset", 2300, 2300)
gc.set_global_step([400, 200, 100])      # Default values from the solution script
#gc.set_range("width", 4300, 4500)       # custom values
#gc.set_range("offset", 2300, 2800)      # custom values
gc.set_range("ext_offset", TARGET_CYCLE - ERROR_TOLERANCE, TARGET_CYCLE + ERROR_TOLERANCE)
#gc.set_range("ext_offset", 40, 200)
#gc.set_range("ext_offset", 1, PROGRAM_MAX_CYCLE_COUNT)
gc.set_step("ext_offset", 1) # We are interested in the whole cycle search space; set step to 1.

scope.glitch.repeat = 1 # This says how many pulses the glitch signal has, not how many times we try. Check docs before touching it!
reboot_flush()
scope.adc.timeout = 1.5 # Number, in seconds, scope will wait before aborting a capture.

hitList = list()        # Holds time and parameters for successful runs
failList = list()       # Holds time and parameters for crashed runs
normalList = list()     # Holds time and parameters for benign runs
counter = 1             # Attempt counter.

clock_ID = time.CLOCK_MONOTONIC # Clock considers time since boot, including time the system has been suspended.
start_time = time.clock_gettime(clock_ID) # Get some time, in seconds. This will be our start time.
end_time = start_time # We will update this value each iteration.

for glitch_settings in gc.glitch_values():
    #start_time = time.time_ns()
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    for i in range(sample_size):
        if scope.adc.state:
            # can detect crash here (fast) before timing out (slow)
            failList.append((end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset))
            gc.add("reset")
            #Device is slow to boot?
            reboot_flush()

        cw.program_target(scope, prog, fw_path)
        scope.arm()
        data = bytearray([0]*5)
        target.simpleserial_write('p', data)
        #target.send_cmd("p", "P", bytearray([0]*5))
        ret = scope.capture()

        # Gather list elements; number of cycles + parameters. ADC counts 4 times each cycle, so we need to divide its count by 4.
        listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
        
        if ret:
            listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
            failList.append(listElement)
            gc.add("reset")
            
            #Device is slow to boot?
            reboot_flush()
        else:
            val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50) #For loop check
            if val['valid'] is False:
                listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                failList.append(listElement)
                gc.add("reset")
            else:

                if val['payload'] == bytearray([1]): #for loop check
                    listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                    hitList.append(listElement)
                    print(end_time-start_time)
                    gc.add("success")
                else:
                    listElement = (end_time - start_time, counter, scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                    normalList.append(listElement)
                    gc.add("normal")        
        end_time = time.clock_gettime(clock_ID) # Update timer
        counter = counter + 1                   # Update interation counter

(ChipWhisperer Target WARNING|File SimpleSerial2.py:385) Unexpected start to command 0x52, expected 0x72
(ChipWhisperer Target WARNING|File SimpleSerial2.py:418) Unexpected length 84, 1
(ChipWhisperer Target WARNING|File SimpleSerial2.py:385) Unexpected start to command 0x52, expected 0x72
(ChipWhisperer Target WARNING|File SimpleSerial2.py:418) Unexpected length 84, 1


In [37]:
# Dump results into .txt files
with open('failList.txt', 'w') as file:
    for item in failList:
        file.write(str(item) + "," + '\n')
with open('hitList.txt', 'w') as file:
    for item in hitList:
        file.write(str(item) + "," + '\n')
with open('normalList.txt', 'w') as file:
    for item in normalList:
        file.write(str(item) + "," + '\n')

In [53]:
hitList

[(129.863245000015, 7, 316905.0, 3500, 2300, 316717),
 (259.1081889999914, 13, 316905.0, 3500, 2300, 316723)]

In [33]:
failList

[(49.34336800000165, 5, 881722.5, 3500, 2300, 463181),
 (61.6382459999877, 6, 1242800.5, 3500, 2300, 463182),
 (61.6382459999877, 6, 10245622.5, 3500, 2300, 463182),
 (87.5328880000161, 7, 10653261.25, 3500, 2300, 463183),
 (87.5328880000161, 7, 919719.5, 3500, 2300, 463183),
 (112.04484899999807, 8, 1389135.25, 3500, 2300, 463184),
 (173.74483099998906, 12, 491650.0, 3500, 2300, 463188),
 (210.6029780000099, 15, 467697.0, 3500, 2300, 463191),
 (259.96496700000716, 19, 845201.5, 3500, 2300, 463195),
 (272.2675870000385, 20, 1163608.75, 3500, 2300, 463196),
 (321.0086040000315, 23, 917406.5, 3500, 2300, 463199),
 (333.3350940000382, 24, 1320105.75, 3500, 2300, 463200),
 (369.96580400003586, 26, 777036.25, 3500, 2300, 463202),
 (369.96580400003586, 26, 846132.5, 3500, 2300, 463202),
 (394.41287400003057, 27, 1302615.75, 3500, 2300, 463203),
 (431.1310999999987, 29, 996098.5, 3500, 2300, 463205),
 (443.5316890000249, 30, 1386359.75, 3500, 2300, 463206),
 (480.6333319999976, 32, 463309.0, 

In [ ]:
normalList #494370.0

In [ ]:
results = gc.calc(ignore_params=["width", "offset"], sort="success_rate")
results

And one for your width/offset settings:

In [ ]:
results = gc.calc(sort="total")
results

In [ ]:
scope.dis()
target.dis()

In [ ]:
assert broken is True

In [3]:
# Target cycle calculated from RTL simulation.
IF_CONDITION = 565637

# Cycles spent bootstrapping and initializing I/O. Chipwhisperer does not count that time. So, subtract that time
MAIN_START = 14339

# Chipwhisperer adds two instructions before the funciton call we are simulating. So, Add that time.
# Also, add time to account for the trigger calls. Exact time is unknown for lowering, assume similar to raising = 45.
CYCLES_FOR_STORE = 6  # Cycles for a storing a word, i.e., 'sw'
CYCLES_FOR_ADD = 2    # Cycles to add an immediate, i.e., 'addi'

# Accruracy for estimations seems to aim a little high for samples on CW, so, adjust it down.
AVERAGE_ERROR = -50.167
OFFSET = 0# int(CYCLES_FOR_STORE + CYCLES_FOR_ADD + AVERAGE_ERROR)


# Note 1: Cycle count found running program and dividing the scope.adc.trig_count, after trigger has been lowered, by 4.
# Note 2: The ADC increments four times each cycle by default. If your numbers are very off, check!
PROGRAM_AVERAGE_CYCLE_COUNT = 463309             # Number of cycles a normal execution takes.
SIMULATION_MAX_CYCLE_COUNT = 565795 - MAIN_START # Number of cycles a simulated execution takes.
SIMULATION_IF_CYCLE_COUNT = IF_CONDITION - MAIN_START + OFFSET
TIME_SCALAR = PROGRAM_AVERAGE_CYCLE_COUNT/SIMULATION_MAX_CYCLE_COUNT # Quotient of the two program runs.
INVERSE_SCALE = SIMULATION_MAX_CYCLE_COUNT/PROGRAM_AVERAGE_CYCLE_COUNT

TARGET_CYCLE = int((SIMULATION_IF_CYCLE_COUNT) * TIME_SCALAR)
ACTUAL = 463160
# Offset start instruction and scale simulated result to match concreate results.
print("Forward  = " + str(TARGET_CYCLE))
print("Backward = " + str(PROGRAM_AVERAGE_CYCLE_COUNT - int(TIME_SCALAR*68) - 55))
print("Actual   = 463160")
print("Error    = " + str(ACTUAL - TARGET_CYCLE))
print("%Error   = " + str(abs(1 - ACTUAL/TARGET_CYCLE)*100))

Forward  = 463176
Backward = 463197
Actual   = 463160
Error    = -16
%Error   = 0.0034544104185019187
